In [0]:
customer_df = spark.read.table('de_mini_project.azure_blob_storage.customer')
display(customer_df)

In [0]:
from pyspark.sql import functions as F
customer_df = customer_df.drop("_file","_line","_modified","_fivetran_synced")

customer_df = customer_df.toDF(
    "customer_id",
    "full_name",
    "contact_info",
    "joined_date",
    "gender_code"
)
#Formatting different dates to YYYY-MM-DD
customer_df = customer_df.withColumn("joined_date",F.coalesce(
        F.try_to_date(F.col("joined_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("joined_date"), "yyyy.MM.dd"),
        F.try_to_date(F.col("joined_date"), "MMM dd, yyyy"),
        F.try_to_date(F.col("joined_date"), "yyyy-MM-dd")
    )
)
#Removing quotes and underscores from name
customer_df = customer_df.withColumn("full_name",F.regexp_replace("full_name",'"',''))
customer_df = customer_df.withColumn("full_name",F.regexp_replace("full_name",'_',' '))
customer_df = customer_df.withColumn("full_name",F.regexp_replace("full_name",' \([^)]*\)',''))
#Removed one record due to it being a duplicate with the only difference being contact number but since raw data was only dates and no timestamp had to remove one
customer_df = customer_df.dropDuplicates(["customer_id"])
customer_df = customer_df.replace("NULL",None)

display(customer_df)

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("de_mini_project.silver.customer")